In [2]:
# sensor_imputation.py
# -------------------------------------------------------
# Stima i sensori mancanti su una cartella di test CSV
# -------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from src.utils import load_config


In [ ]:
# -------------------------------------------------------
# CONFIGURAZIONE
# -------------------------------------------------------
cfg = load_config('./configs/config.yaml')

TRAIN_PATH      = cfg['data']['train_clean_csv']
TEST_FOLDER     = cfg['data']['test']
OUTPUT_FOLDER   = cfg['data']['test_imputed']

SENSORS_TO_IMPUTE = ["Sensed_P25", "Sensed_T5"]  

EXCLUDE_COLS = ["ESN", "Cycles", "Cycles_to_WW",
                "Cycles_to_HPC_SV", "Cycles_to_HPT_SV",'HPC_Eff_Index_clean','ratio_T3_T45','tri_ratio_diff_Sensed_Mach_Sensed_T3_Sensed_T45'] + SENSORS_TO_IMPUTE


In [ ]:
# -------------------------------------------------------
# 1. CARICAMENTO TRAINING
# -------------------------------------------------------
train_df = pd.read_csv(TRAIN_PATH)
print(f"Train shape: {train_df.shape}")


In [ ]:
# -------------------------------------------------------
# 2. FEATURE PER IL REGRESSORE AUSILIARIO
# -------------------------------------------------------
# Carica un file test di esempio per verificare le colonne disponibili
sample_test = pd.read_csv(next(Path(TEST_FOLDER).glob("test_*.csv")))

available_features = [
    c for c in train_df.columns
    if c not in EXCLUDE_COLS and c in sample_test.columns
]

print(f"Feature usate per imputazione ({len(available_features)}): {available_features}")


In [ ]:
# -------------------------------------------------------
# 3. ADDESTRAMENTO REGRESSORI (una volta sola sul train)
# -------------------------------------------------------
imputers = {}

for sensor in SENSORS_TO_IMPUTE:
    if sensor not in train_df.columns:
        print(f"[WARN] {sensor} non trovato nel training — skip.")
        continue

    # Droppa righe con NaN nel target o nelle feature, solo per questo fit
    train_clean = train_df[available_features + [sensor]].dropna(subset=[sensor] + available_features)
    print(f"[INFO] {sensor}: {len(train_df)} righe totali -> {len(train_clean)} dopo dropna")

    model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
    model.fit(train_clean[available_features], train_clean[sensor])

    y_pred_train = model.predict(train_clean[available_features])
    mae = mean_absolute_error(train_clean[sensor], y_pred_train)
    r2  = r2_score(train_clean[sensor], y_pred_train)

    imputers[sensor] = model
    print(f"[TRAIN] {sensor} -> MAE: {mae:.4f}, R²: {r2:.4f}")



In [ ]:
# -------------------------------------------------------
# 4. ELABORAZIONE DI OGNI FILE TEST
# -------------------------------------------------------
test_files = sorted(Path(TEST_FOLDER).glob("test_*.csv"))
output_dir = Path(OUTPUT_FOLDER)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\nFile test trovati: {len(test_files)}")

for test_path in test_files:
    test_df = pd.read_csv(test_path)

    ## --- Clamping outlier con bound del train ---
    #for col, (lo, hi) in clamp_bounds.items():
    #    if col in test_df.columns:
    #        test_df[col] = test_df[col].clip(lo, hi)
#
    # --- Imputazione sensori mancanti ---
    for sensor, model in imputers.items():
        test_df[sensor] = model.predict(test_df[available_features])

    # --- Salvataggio ---
    out_path = output_dir / test_path.name
    test_df.to_csv(out_path, index=False)
    print(f"[OK] {test_path.name} -> {test_df.shape} -> salvato in {out_path}")

print(f"\nDone. {len(test_files)} file salvati in '{OUTPUT_FOLDER}'")
